In [1]:
from src.gaussian_mixture import GaussianMixtureMAR
from src.dg import DataGenerator

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, logistic, t

import torch

from joblib import Parallel, delayed

Helper functions

In [2]:
# functions to numerically find the quantile of a 1D Gaussian mixture
def gmm_cdf(x, mu, sigma, pi):
    z = (x - mu) / sigma
    Phi = 0.5 * (1 + torch.erf(z / torch.sqrt(torch.tensor(2., device=z.device))))
    return (pi * Phi).sum()

def gmm_quantile(p, mu, sigma, pi, tol=1e-6, max_iter=1000):
    """
    Compute x such that P(X <= x) = p for a 1D Gaussian mixture.
    use binary search (bisection) to numerically find x.
    """
    lo = (mu - 10 * sigma).min()
    hi = (mu + 10 * sigma).max()

    for _ in range(max_iter):
        mid = (lo + hi) / 2
        if gmm_cdf(mid, mu, sigma, pi) < p:
            lo = mid
        else:
            hi = mid

        if (hi - lo) < tol:
            break

    return (lo + hi) / 2

def true_quantile(dist, p=0.1, mu=None, sigma=None, pi=None, tol=1e-6, max_iter=100, df=None):
    if dist == 'Normal':
        return norm.ppf(p)
    elif dist == 'gaussian_mixture' and all(x is not None for x in (mu, sigma, pi)):
        return gmm_quantile(p, mu, sigma, pi, tol, max_iter)
    elif dist == 'Logistic':
        return logistic.ppf(p)
    elif dist == 'Student t':
        return t.ppf(p, df)

def quantile_x1(model, p, n_samples):
    x = model.sample(n_samples)[0]
    x1 = x[:, 0]
    q = np.quantile(x1, p)
    return q

def worker(delta):
    dg = DataGenerator(distr=dist, alpha=alpha, delta=delta)
    X, M = dg.generate(N, d)
    gmm.fit(X, M)
    return quantile_x1(gmm, p, n_samples)

In [3]:
# global parameters
N = 5000
n_samples = 5000
d = 3
cov_type = 'diag'
k_range = range(2, 9)
p = 0.1

B = 50

gmm = GaussianMixtureMAR(
    k_range=k_range,
    criterion='bic',
    device='cpu',
    cov_type=cov_type,
    n_init=20
    )

Normal with 
$$\Sigma = \begin{bmatrix} 1 & 0.7 & 0 \\ 0.7 & 1 & 0 \\ 0 & 0 & 1 \end{bmatrix}$$

In [8]:
# param_vals
dist = 'Normal'
alpha = 0.7

deltas = [0.05 * x for x in range(1, 10)]

In [ ]:
res = Parallel(n_jobs=15)(
        delayed(worker)(delta) 
        for delta in deltas
        for _ in range(B)
    )

In [ ]:
delta_invs = [round(1/x ** 0.5,1) for x in deltas]

In [ ]:
plt.boxplot(qqs.T, tick_labels=delta_invs)
plt.axhline(y=true_quantile(dist), color='blue', linestyle='--')
plt.ylabel(r"$F^{-1} (0.1)$")
plt.xlabel(r'$\frac{1}{\sqrt{\delta}}$')
plt.show()